# 固定方案的实验总表

汇总已经完成的验证集和测试集结果，并记录各方案在测试前的定位。此处只读取已有结果，不重新训练、不搜索参数。


In [ ]:
# 加载已保存的实验结果与固定测试协议
import hashlib
import json
import numpy as np
import pandas as pd

from sentiment_analysis.paths import ROOT, TABLES_DIR

protocol_path = ROOT / "configs" / "final_test_protocol.json"
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
expected_protocol_hash = "ae5c7db09e1e300f700ef9b8648d378ec6392496df5ab14d932020d9b93bb876"
if hashlib.sha256(protocol_path.read_bytes()).hexdigest() != expected_protocol_hash:
    raise ValueError("Fixed test protocol has changed.")

traditional_results = pd.read_csv(TABLES_DIR / "traditional_ml_validation_metrics.csv")
single_results = pd.read_csv(TABLES_DIR / "lstm_validation_comparison.csv")
unidirectional_ensemble = pd.read_csv(TABLES_DIR / "textcnn_lstm_ensemble_comparison.csv")
bidirectional_ensemble = pd.read_csv(TABLES_DIR / "textcnn_bilstm_ensemble_comparison.csv")
test_results = pd.read_csv(TABLES_DIR / "fixed_models_test_results.csv")


In [ ]:
# 按固定方案对齐验证集和测试集指标
metrics = ["ACCURACY", "PRECISION", "RECALL", "F1"]

def select_one(data, mask, name):
    selected = data.loc[mask]
    if len(selected) != 1:
        raise ValueError(f"Expected one result for {name}, got {len(selected)}.")
    return selected.iloc[0]

validation_rows = [
    ("B06", "Traditional baseline", "Combined TF-IDF, LinearSVC C=1.0",
     select_one(traditional_results, traditional_results["EXPERIMENT"] == "B06", "B06")),
    ("TextCNN", "Single-model comparator", "Kernel 3, seed 42, epoch 6",
     select_one(single_results, single_results["EXPERIMENT"] == "TextCNN", "TextCNN")),
    ("LSTM", "Single-model comparator", "Unidirectional, seed 42, epoch 3",
     select_one(single_results, single_results["EXPERIMENT"] == "LSTM", "LSTM")),
    ("BiLSTM", "Single-model comparator", "Bidirectional, seed 42, epoch 3",
     select_one(bidirectional_ensemble, (bidirectional_ensemble["SEED"] == 42) & (bidirectional_ensemble["MODEL"] == "BiLSTM"), "BiLSTM")),
    ("TextCNN + LSTM (equal)", "Primary candidate", "Seed 42, 0.5 / 0.5",
     select_one(unidirectional_ensemble, unidirectional_ensemble["MODEL"] == "Equal-weight ensemble", "Unidirectional ensemble")),
    ("TextCNN + BiLSTM (equal)", "Ensemble comparator", "Seed 42, 0.5 / 0.5",
     select_one(bidirectional_ensemble, (bidirectional_ensemble["SEED"] == 42) & (bidirectional_ensemble["MODEL"] == "Equal-weight ensemble"), "Bidirectional ensemble")),
]

rows = []
for name, role, configuration, validation_row in validation_rows:
    test_row = select_one(test_results, test_results["MODEL"] == name, name + " test")
    row = {"MODEL": name, "ROLE": role, "CONFIGURATION": configuration}
    for metric in metrics:
        row["VALIDATION " + metric] = validation_row[metric]
        row["TEST " + metric] = test_row[metric]
    rows.append(row)

experiment_summary = pd.DataFrame(rows)
display(experiment_summary.round(4))


In [ ]:
# 保存完整指标，并检查测试方案与协议一致
expected_models = {item[0] for item in validation_rows}
if set(test_results["MODEL"]) != expected_models:
    raise ValueError("Test results do not match the selected fixed models.")
if len(experiment_summary) != len(expected_models):
    raise ValueError("Unexpected number of summary rows.")

output_path = TABLES_DIR / "fixed_experiment_summary.csv"
experiment_summary.to_csv(output_path, index=False)
print(f"Summary saved: {output_path}")


## 当前研究问题与证据

1. 不同方法的表现：已有相同清洗后数据划分上的 TF-IDF、CNN、LSTM 及融合结果；模型预处理和参数规模不同，不能将差异只归因于模型结构。
2. CNN 与 LSTM 互补性：验证集上统计过预测交叉、纠正与新增错误，并测试过等权和加权软投票；目前的样例只能辅助解释，不能证明错误原因。
3. 融合效果：测试前固定的主方案为 CNN + 单向 LSTM 等权融合，双向融合为对照。两种方案在同一测试集上高于对应单模型，但均低于 B06；结果不用于再次选择或调整方案。
4. DistilBERT 与效率：尚未完成 DistilBERT 微调和统一的计算效率实验，暂时无法回答有关预训练模型与融合方案的完整权衡。
5. 消融：已有特征与结构对照，但部分实验同时改变了词表组成或参数规模，不应全部称为严格消融。后续需要先确定问题与控制变量。

验证集已反复用于探索与选择；测试集结果也已查看。后续实验应如实区分探索、固定方案的评估和任何新方案的评估。清洗后的测试集不等同于完整的官方 IMDB 测试集。
